### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** You don't just 'use a transformer'. You understand the trade-offs: **RoPE** for context length extrapolation, **GQA** for multi-user inference throughput, and **Pre-Norm** for training stability in 100+ layer architectures. You know that attention is not 'magic' — it's just a way to dynamically route information based on relevance.

**Mental Model:** Imagine a library where every book (token) has a post-it note. 
- **Query:** Your search term.
- **Key:** The index on the shelf.
- **Value:** The actual text in the book.
- **GQA:** Sharing the same index/text for multiple search terms to save bookshelf space.
- **RoPE:** Giving every book a 'clock' that rotates based on its position, so the model 'feels' the distance between them.

**Common Junior Pitfall:** Recomputing the entire sequence's attention for every single token generated. Seniors use a **KV-Cache** to store past results, making generation linear in time rather than quadratic.

---


## 1. Self-Attention (Mechanism)
At its heart, attention is: $\text{softmax}(\frac{QK^T}{\sqrt{d_k}})V$.

### 🎓 Socratic Deep Check
> *"Why do we divide by $\sqrt{d_k}$? If we didn't, and $d_k=1024$, how would it change the 'sharpness' of the softmax distribution? What would that do to the gradients during backpropagation?"*

## 📑 Table of Contents

* [🎓 Socratic Deep Check](#-socratic-deep-check)
  * [🎓 Junior to Senior: Concept Bridge](#-junior-to-senior-concept-bridge)
* [1. Self-Attention (Mechanism)](#1-self-attention-mechanism)
  * [🎓 Socratic Deep Check](#-socratic-deep-check)
* [2. Grouped-Query Attention (GQA)](#2-grouped-query-attention-gqa)
* [3. Rotary Position Embeddings (RoPE) — Complete Derivation](#3-rotary-position-embeddings-rope-complete-derivation)
* [4. Pre-Norm vs Post-Norm ARCHITECTURE](#4-pre-norm-vs-post-norm-architecture)
* [5. The KV-Cache — Why Autoregressive Generation Is Not $O(n^2)$](#5-the-kv-cache-why-autoregressive-generation-is-not-on2)
* [✅ Knowledge Check](#-knowledge-check)
  * [Q1: Why use GQA instead of MHA?](#q1-why-use-gqa-instead-of-mha)
  * [Q2: Absolute vs Relative Positional Encoding](#q2-absolute-vs-relative-positional-encoding)
* [🔨 Practical Practice](#-practical-practice)
  * [Tier 1: Beginner](#tier-1-beginner)
  * [Tier 2: Intermediate](#tier-2-intermediate)
  * [Tier 3: Advanced](#tier-3-advanced)

---


In [ ]:
import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(q, k, v, mask=None, dropout=None):
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    p_attn = F.softmax(scores, dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, v), p_attn

"""
What just happened?
We implemented the core Attention formula. Notice the mask filling with -infinity
(effectively -1e9) — this ensures 'future' tokens contribute exactly 0 weight 
to the softmax, preventing the model from 'cheating' during text generation.
"""

## 2. Grouped-Query Attention (GQA)

Multi-Head Attention (MHA) has $H$ query heads, $H$ key heads, and $H$ value heads. While parallelized, this kills memory because every user session must store $H$ KV-states per token.

**The Innovation:** 
- **MQA (Multi-Query):** 1 KV-head for all Q-heads. Efficient, but loses quality.
- **GQA (Grouped-Query):** $G$ KV-heads for $H$ Q-heads. Best of both worlds.

📈 **Production Signal:** **Llama 3 8B** uses $H=32, G=8$. For a 32K context window, GQA reduces KV-cache memory by **4x**, making high-throughput inference possible on consumer GPUs.


In [ ]:
class ModernMultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, num_heads, num_kv_heads=None):
        super().__init__()
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads if num_kv_heads is not None else num_heads
        self.head_dim = d_model // num_heads
        self.kv_group_size = self.num_heads // self.num_kv_heads
        
        self.q_proj = torch.nn.Linear(d_model, num_heads * self.head_dim, bias=False)
        self.k_proj = torch.nn.Linear(d_model, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = torch.nn.Linear(d_model, self.num_kv_heads * self.head_dim, bias=False)
        self.o_proj = torch.nn.Linear(num_heads * self.head_dim, d_model, bias=False)
        
    def forward(self, x, mask=None):
        B, L, _ = x.shape
        q = self.q_proj(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, L, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, L, self.num_kv_heads, self.head_dim).transpose(1, 2)
        
        # GQA Trick: Repeat KV heads to match Q-heads
        if self.num_kv_heads != self.num_heads:
            k = torch.repeat_interleave(k, self.kv_group_size, dim=1)
            v = torch.repeat_interleave(v, self.kv_group_size, dim=1)
        
        attn_out, _ = scaled_dot_product_attention(q, k, v, mask)
        attn_out = attn_out.transpose(1, 2).reshape(B, L, -1)
        return self.o_proj(attn_out)

"""
What just happened?
We implemented Grouped-Query Attention. If num_kv_heads is less than num_heads, 
we 'broadcast' the KV keys/values across multiple Query heads. This preserves 
accuracy while drastically reducing the size of the weights and the KV-cache.
"""

## 3. Rotary Position Embeddings (RoPE) — Complete Derivation

Original transformers used absolute sine waves $x + PE$. Modern models use **RoPE**, which rotates vectors in complex space.

**The Math:**
For a dimension pair $(2i, 2i+1)$ at position $m$:
$$R_m = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix}$$
Where $\theta_i$ is a base frequency $10000^{-2i/d}$.

**The Critical Relative Property:**
The dot product of two rotated vectors $(R_m q)^T (R_n k)$ is mathematically equal to $q^T R_{n-m} k$. 

This means the model doesn't need to know 'Where is token X'? It only needs to know 'How far is token X from token Y?'. This allows Llama to extrapolate to much longer context windows than it was trained on.

📈 **Production Signal:** RoPE is the industry standard for **context window expansion** (e.g. going from 8K training to 128K inference windows).


In [ ]:
def apply_rotary_emb(x, pos_ids, base=10000):
    # Simple implementation of RoPE rotation logic
    d = x.shape[-1]
    # Compute frequencies
    theta = 1.0 / (base ** (torch.arange(0, d, 2).float() / d))
    m_theta = pos_ids.unsqueeze(-1) * theta.unsqueeze(0) # (seq, d/2)
    
    cos = torch.cos(m_theta).unsqueeze(-2) # (seq, 1, d/2)
    sin = torch.sin(m_theta).unsqueeze(-2) # (seq, 1, d/2)
    
    # Split input into even/odd dimension pairs and rotate
    x1 = x[..., 0::2]
    x2 = x[..., 1::2]
    
    # RoPE rotation formula: [x1*cos - x2*sin, x2*cos + x1*sin]
    x_rope = torch.stack([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)
    return x_rope.flatten(-2)

"""
What just happened?
We applied RoPE. Instead of adding a value to the embeddings, we effectively 
twisted each vector by an angle proportional to its position. This 'rotational signal' 
is preserved through the linear projections and dot products.
"""

## 4. Pre-Norm vs Post-Norm ARCHITECTURE

How do we layer our Normalization? 
- **Post-Norm (Original):** $x = LN(x + \text{Sublayer}(x))$ 
- **Pre-Norm (Modern):** $x = x + \text{Sublayer}(LN(x))$

**Why LLMs use Pre-Norm:**
In Post-Norm, gradients must pass through many LayerNorm units, which can lead to instability as the network deepens. Pre-Norm leaves the "Residual Highway" ($x = x + \dots$) untouched at its core, making training 100+ layer transformers much more stable.

📈 **Production Signal:** **GPT-3** and **Llama** use Pre-Norm. **GPT-2** was the first major model to demonstrate its superiority for scaling.


In [ ]:
class TransformerBlock(torch.nn.Module):
    def __init__(self, d_model, num_heads, norm_type='pre'):
        super().__init__()
        self.attn = ModernMultiHeadAttention(d_model, num_heads)
        self.norm1 = torch.nn.LayerNorm(d_model)
        self.norm2 = torch.nn.LayerNorm(d_model)
        self.ffn = torch.nn.Sequential(
            torch.nn.Linear(d_model, 4 * d_model), 
            torch.nn.GELU(), 
            torch.nn.Linear(4 * d_model, d_model)
        )
        self.norm_type = norm_type
        
    def forward(self, x, mask=None):
        if self.norm_type == 'pre':
            x = x + self.attn(self.norm1(x), mask)
            x = x + self.ffn(self.norm2(x))
        else:
            x = self.norm1(x + self.attn(x, mask))
            x = self.norm2(x + self.ffn(x))
        return x

"""
What just happened?
We implemented both variants. Notice that Pre-Norm allows the original signal 'x' 
to survive all the way to the output with its identity preserved. This architectural 
difference is why 2024 models are so much more stable than 2017 models.
"""

## 5. The KV-Cache — Why Autoregressive Generation Is Not $O(n^2)$

Generating text is **iterative**: you generate Token 1, then Token 1+2, then Token 1+2+3. 

If you re-compute everything at every step, your compute cost grows quadratically. 
**The KV-Cache** stores the computed $K$ and $V$ vectors for all previous tokens. At each new step, we only compute $Q, K, V$ for the ONE new token, and append $K, V$ to our cache memory. 

This turns generation into **O(n)** time complexity. 

📈 **Production Signal:** When a GPU 'runs out of memory' during long chat sessions, it's usually because the **KV-Cache** has filled up the entire VRAM (e.g., 40,000 tokens for 70B model requires ~100GB of cache).


In [ ]:
import time

def generation_benchmark(seq_len=200):
    # Simulation of compute cost
    # Naive: n + (n+1) + (n+2) ...
    naive_cost = sum(range(seq_len))
    # KV-Cache: 1 + 1 + 1 ...
    cache_cost = seq_len
    
    print(f"For sequence length {seq_len}:")
    print(f"  Total Matrix Multiplications (Naive): {naive_cost:,}")
    print(f"  Total Matrix Multiplications (Cache): {cache_cost:,}")
    print(f"  Theoretical Speedup: {naive_cost / cache_cost:.1f}x")

generation_benchmark()

"""
What just happened?
We de-mystified why LLMs are fast enough to chat with. Without this O(n) optimization,
generating a long 2000-word essay would be 1000x slower at the end than at the start.
"""

---
## ✅ Knowledge Check

### Q1: Why use GQA instead of MHA?
<details><summary>👉 Answer</summary>
GQA significantly reduces the memory required for the KV-cache. Since LLM inference is often bottlenecked by VRAM capacity and memory bandwidth (not FLOPs), reducing the KV heads allows for much larger batch sizes and faster generation across many users simultaneously.
</details>

### Q2: Absolute vs Relative Positional Encoding
<details><summary>👉 Answer</summary>
Absolute encoding (sinusoidal) says 'I am position 5'. If the model never saw position 5000 in training, it doesn't know what to do. RoPE (relative) allows the model to learn that 'I am 5 steps away from you'. This geometry works at any distance, facilitating 'Extrapolation' to longer sequences.
</details>


---
## 🔨 Practical Practice

### Tier 1: Beginner
1. Implement a 2D vector rotation using a rotation matrix and verify it preserves the length (norm) of the vector.
2. Modify `scaled_dot_product_attention` to print the attention weights before and after the causal mask is applied.

### Tier 2: Intermediate
1. **KV-Cache Math:** Calculate total KV-cache memory for **Llama 3 8B** (32 heads, 32 layers, head_dim 128, float16). How much VRAM is needed for 8192 tokens? 
2. **RoPE Extrapolation:** Verify the relative property: rotate three vectors q, k1, k2 such that k1 is at dist 2 from q, and k2 is at dist 2 from q (at different absolute positions). Show dot(q_rot, k1_rot) == dot(q_rot, k2_rot).

### Tier 3: Advanced
1. **The Flash Prelude:** Implement the attention operation using only tiling and localized memory (simulate what Flash Attention does). Prove that you never need to store the large $N \times N$ attention matrix in full memory at any time.
2. **MiniGPT Modernization:** Modify the `MiniGPT` class to use `ModernMultiHeadAttention` with GQA=4 and RoPE. Train it on a small char-dataset and compare its generation quality to the original absolute-encoding version.
